# Traffic Pattern Analysis for Smart Cities
This notebook contains the data analysis and machine learning models required for predicting traffic volumes, identifying congestion hotspots, and forecasting travel times.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

## 1. Mock Data Generation
Since we don't have access to real IoT sensor data, we will generate a synthetic dataset mimicking real-time traffic data.

In [ ]:
np.random.seed(42)
num_records = 5000
start_time = datetime.now() - timedelta(days=30)

# Generate timestamps
timestamps = [start_time + timedelta(minutes=15 * i) for i in range(num_records)]

# Generate locations (e.g., 5 major intersections)
locations = ['Intersection A', 'Intersection B', 'Highway 1 Exit', 'Downtown Hub', 'Suburban Route']
loc_data = np.random.choice(locations, num_records)

# Generate weather (0: Clear, 1: Rain, 2: Snow)
weather = np.random.choice([0, 1, 2], num_records, p=[0.7, 0.2, 0.1])

# Generate base traffic volumes and speeds (incorporating daily patterns)
volumes = []
speeds = []
for i, ts in enumerate(timestamps):
    hour = ts.hour
    # Peak hours: 7-9 AM, 5-7 PM
    if (7 <= hour <= 9) or (17 <= hour <= 19):
        base_vol = np.random.normal(500, 50)
        base_speed = np.random.normal(20, 5) # Lower speed during peak
    else:
        base_vol = np.random.normal(200, 30)
        base_speed = np.random.normal(45, 10)
    
    # Weather impact
    if weather[i] == 1: # Rain
        base_vol *= 0.9
        base_speed *= 0.8
    elif weather[i] == 2: # Snow
        base_vol *= 0.7
        base_speed *= 0.6
        
    volumes.append(max(0, int(base_vol)))
    speeds.append(max(0, base_speed))

# Create DataFrame
df = pd.DataFrame({
    'timestamp': timestamps,
    'location': loc_data,
    'volume': volumes,
    'speed_kmh': speeds,
    'weather_condition': weather
})

df.head()

## 2. Congestion Hotspot Identification (Clustering)
We use K-Means clustering to categorize traffic states into different congestion levels (e.g., Low, Medium, High).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# We'll cluster based on volume and speed
X_cluster = df[['volume', 'speed_kmh']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Apply K-Means
kmeans = KMeans(n_clusters=3, random_state=42)
df['congestion_cluster'] = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='volume', y='speed_kmh', hue='congestion_cluster', palette='viridis', alpha=0.6)
plt.title('Traffic Congestion Clusters')
plt.xlabel('Traffic Volume')
plt.ylabel('Average Speed (km/h)')
plt.show()


## 3. Time Series Forecasting (ARIMA)
We will use an ARIMA model to predict future traffic volumes for a specific location.

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error

# Filter data for one specific intersection and aggregate by hour
ts_df = df[df['location'] == 'Downtown Hub'].copy()
ts_df.set_index('timestamp', inplace=True)
hourly_vol = ts_df['volume'].resample('h').mean().ffill()

# Train-test split (80/20)
train_size = int(len(hourly_vol) * 0.8)
train, test = hourly_vol[:train_size], hourly_vol[train_size:]

# Fit ARIMA model (Parameters p,d,q would normally be tuned via grid search)
model = ARIMA(train, order=(2, 1, 2))
model_fit = model.fit()

# Predict
predictions = model_fit.forecast(steps=len(test))

plt.figure(figsize=(12, 5))
plt.plot(train.index[-100:], train.values[-100:], label='Train')
plt.plot(test.index, test.values, label='Actual Test')
plt.plot(test.index, predictions, color='red', label='ARIMA Predictions')
plt.title('ARIMA Traffic Volume Forecast')
plt.legend()
plt.show()

## 4. Congestion Prediction (Random Forest)
We build a predictive model to classify the expected congestion cluster based on time of day, location, and weather.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Prepare features
df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek

features = ['hour', 'day_of_week', 'weather_condition']
X = df[features]
y = df['congestion_cluster']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print(classification_report(y_test, y_pred))

## 5. Sequence Prediction (LSTM)
Using Long Short-Term Memory (LSTM) networks to predict sequence patterns for traffic speed.

In [ ]:
# Note: Make sure tensorflow/keras is installed
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

scaler_lstm = MinMaxScaler(feature_range=(0, 1))
scaled_speed = scaler_lstm.fit_transform(hourly_vol.values.reshape(-1, 1))

def create_dataset(dataset, look_back=12):
    X, Y = [], []
    for i in range(len(dataset)-look_back-1):
        a = dataset[i:(i+look_back), 0]
        X.append(a)
        Y.append(dataset[i + look_back, 0])
    return np.array(X), np.array(Y)

look_back = 12 # Look back 12 hours
X_lstm, Y_lstm = create_dataset(scaled_speed, look_back)
X_lstm = np.reshape(X_lstm, (X_lstm.shape[0], X_lstm.shape[1], 1))

train_size_lstm = int(len(X_lstm) * 0.8)
X_train_lstm, X_test_lstm = X_lstm[0:train_size_lstm], X_lstm[train_size_lstm:]
Y_train_lstm, Y_test_lstm = Y_lstm[0:train_size_lstm], Y_lstm[train_size_lstm:]

model_lstm = Sequential()
model_lstm.add(LSTM(50, return_sequences=True, input_shape=(look_back, 1)))
model_lstm.add(LSTM(50))
model_lstm.add(Dense(1))
model_lstm.compile(loss='mean_squared_error', optimizer='adam')

# Train model (uncomment to train, using epochs=10 for speed)
print("Training LSTM model...")
model_lstm.fit(X_train_lstm, Y_train_lstm, epochs=5, batch_size=32, verbose=0)
print("LSTM Model Training Complete.")

lstm_predictions = model_lstm.predict(X_test_lstm)
lstm_predictions = scaler_lstm.inverse_transform(lstm_predictions)
actual_values = scaler_lstm.inverse_transform(Y_test_lstm.reshape(-1, 1))

plt.figure(figsize=(10,5))
plt.plot(actual_values[:100], label='Actual Speed')
plt.plot(lstm_predictions[:100], color='red', label='LSTM Predicted')
plt.title('LSTM Traffic Speed Prediction')
plt.legend()
plt.show()